In [4]:
import os
import cv2
import torch
import random
from torch.utils.data import Dataset, DataLoader

# 1. Read the manifest
data_entries = []
with open("../data/processed/crnn_training_data/labels.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            data_entries.append((parts[0], parts[1]))

# 2. Build the Character Map
unique_chars = set()
for _, text in data_entries:
    unique_chars.update(list(text))

chars = sorted(list(unique_chars))
char2idx = {char: idx + 1 for idx, char in enumerate(chars)} # +1 to leave 0 for CTC Blank
idx2char = {idx + 1: char for idx, char in enumerate(chars)}

print(f"Vocabulary size: {len(chars)} characters + 1 CTC Blank")

Vocabulary size: 28 characters + 1 CTC Blank


In [5]:
class GreekSqueezeDataset(Dataset):
    def __init__(self, entries, img_dir, char2idx):
        self.entries = entries
        self.img_dir = img_dir
        self.char2idx = char2idx

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        filename, text = self.entries[idx]
        
        # Load image in grayscale (CRNNs usually prefer 1-channel input)
        img_path = os.path.join(self.img_dir, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # Normalize pixel values to [-1, 1]
        img = (img / 127.5) - 1.0
        
        # Shape becomes (Channels=1, Height=32, Width=W)
        img_tensor = torch.FloatTensor(img).unsqueeze(0) 
        
        # Convert string to list of integers
        target = [self.char2idx[c] for c in text]
        
        return img_tensor, target

# --- RUNTIME SPLITTING ---
random.shuffle(data_entries)
split_idx = int(len(data_entries) * 0.8) # 80% Train, 20% Val

train_dataset = GreekSqueezeDataset(data_entries[:split_idx], "../data/processed/crnn_training_data/images", char2idx)
val_dataset = GreekSqueezeDataset(data_entries[split_idx:], "../data/processed/crnn_training_data/images", char2idx)

print(f"Training samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")

Training samples: 7054 | Validation samples: 1764


In [6]:
def crnn_collate_fn(batch):
    images, targets = zip(*batch)
    
    # Find the max width in this specific batch
    max_width = max([img.shape[2] for img in images])
    
    padded_images = []
    for img in images:
        c, h, w = img.shape
        # Pad right side with 0 (which is -1.0 in our normalized scale)
        pad_amount = max_width - w
        padded_img = torch.nn.functional.pad(img, (0, pad_amount, 0, 0), value=-1.0)
        padded_images.append(padded_img)
        
    images_tensor = torch.stack(padded_images)
    
    # CTC Loss expects a flat 1D tensor of all targets, 
    # plus a separate tuple telling it how long each target is
    target_lengths = torch.IntTensor([len(t) for t in targets])
    targets_flat = torch.IntTensor([char_idx for t in targets for char_idx in t])
    
    return images_tensor, targets_flat, target_lengths

# Create the DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=crnn_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=crnn_collate_fn)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GreekCRNN(nn.Module):
    def __init__(self, img_channels=1, num_chars=35, hidden_size=256):
        super(GreekCRNN, self).__init__()
        
        # 1. CNN Feature Extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(img_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # H: 32 -> 16
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # H: 16 -> 8
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)), # H: 8 -> 4, Width preserved
            
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)), # H: 4 -> 2, Width preserved
            
            nn.Conv2d(512, 512, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True) 
            # Final Output Height: 1. Width: Original_Width / 4 - 1
        )
        
        # 2. Sequence Modeler (Bi-LSTM)
        self.rnn = nn.LSTM(input_size=512, 
                           hidden_size=hidden_size, 
                           num_layers=2, 
                           bidirectional=True, 
                           batch_first=True)
        
        # 3. Transcription Layer
        # +1 is mandatory for the CTC "Blank" token (Index 0)
        self.fc = nn.Linear(hidden_size * 2, num_chars + 1)

    def forward(self, x):
        # Pass through CNN: [Batch, 1, 32, Width] -> [Batch, 512, 1, Seq_Len]
        conv_out = self.cnn(x)
        
        # Squeeze height to 1D and permute for RNN: -> [Batch, Seq_Len, 512]
        conv_out = conv_out.squeeze(2).permute(0, 2, 1) 
        
        # Pass through Bi-LSTM: -> [Batch, Seq_Len, 512]
        rnn_out, _ = self.rnn(conv_out)
        
        # Get character probabilities: -> [Batch, Seq_Len, Num_Chars + 1]
        out = self.fc(rnn_out)
        
        return out

print("CRNN Architecture Defined.")

CRNN Architecture Defined.


In [8]:
# 1. Hardware setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# 2. Initialize Model, Loss, and Optimizer
# Ensure len(chars) matches the vocabulary size you built in the previous step
model = GreekCRNN(img_channels=1, num_chars=len(chars), hidden_size=256).to(device)

# zero_infinity=True prevents the loss from exploding to NaN if a target string 
# is accidentally longer than the CNN's output sequence length.
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 3. Quick 1-Epoch Validation Loop
model.train()
print("Starting 1-Epoch Validation Test...")

for batch_idx, (images, targets, target_lengths) in enumerate(train_loader):
    images = images.to(device)
    targets = targets.to(device)
    target_lengths = target_lengths.to(device)
    
    optimizer.zero_grad()
    
    # Forward pass: [Batch, Seq_Len, Num_Classes]
    outputs = model(images)
    
    # CTCLoss expects log probabilities
    outputs = F.log_softmax(outputs, dim=2)
    
    # Permute to [Seq_Len, Batch, Num_Classes]
    outputs = outputs.permute(1, 0, 2)
    
    # Calculate sequence lengths for this specific batch
    # Since our collate_fn padded all images to the same width, 
    # every sequence in this batch has the same length.
    seq_len = outputs.size(0)
    batch_size = images.size(0)
    input_lengths = torch.full(size=(batch_size,), fill_value=seq_len, dtype=torch.long).to(device)
    
    # Calculate Loss
    loss = criterion(outputs, targets, input_lengths, target_lengths)
    
    # Backward pass and optimize
    loss.backward()
    optimizer.step()
    
    # Print progress every 10 batches
    if batch_idx % 10 == 0:
        print(f"Batch {batch_idx}/{len(train_loader)} | CTC Loss: {loss.item():.4f}")

print("Validation Epoch Complete! Pipeline is flowing correctly.")

Training on: cpu
Starting 1-Epoch Validation Test...
Batch 0/441 | CTC Loss: 66.8287
Batch 10/441 | CTC Loss: 6.3878
Batch 20/441 | CTC Loss: 3.3716
Batch 30/441 | CTC Loss: 3.5106


: 